# 04 — FLORES-200 replication & downstream transfer
Two additional experiments for the EACL 2027 paper:

**A. FLORES-200 fertility replication.** Re-run the tokenizer fertility audit on FLORES-200 devtest (1,012 sentences, same languages) and check that per-tokenizer inequality (p90/p10) and per-language fertility replicate. This answers the reviewer objection "FLORES-200 is also multi-parallel, why not use it?" with data: the audit replicates on FLORES, and the Qur'an corpus additionally supplies what FLORES cannot (retrieval gold pairs at scale, independent non-English source).

**B. Downstream zero-shot transfer.** Meccan vs Medinan verse classification: train a logistic-regression probe on English verse embeddings, evaluate zero-shot on the same held-out verses in every other language (labels transfer through the verse alignment). Correlate the per-language transfer score with the retrieval axis and the fertility axis.

Runtime: A ≈ 10 min (CPU ok). B ≈ 20 min on a T4 GPU for all four encoders.


In [ ]:
!pip -q install tiktoken transformers sentencepiece sentence-transformers scikit-learn scipy pandas matplotlib
# Optional: log in to use the official gated tokenizers (meta-llama/Meta-Llama-3-8B, google/gemma-2-9b).
# Without login, the ungated mirror copies below are used and give identical token counts.
# from huggingface_hub import notebook_login; notebook_login()

In [ ]:
# Upload translations.zip (the 41-file corpus) — or mount Drive and point CORPUS_ZIP at it.
import os
CORPUS_ZIP = "translations.zip"
if not os.path.exists(CORPUS_ZIP):
    from google.colab import files
    up = files.upload()
    CORPUS_ZIP = list(up)[0]
!unzip -qo {CORPUS_ZIP} -d corpus
!ls corpus | wc -l

In [ ]:
# FLORES-200 devtest (official tarball, CC-BY-SA 4.0)
!wget -q https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz && tar xzf flores200_dataset.tar.gz
!ls flores200_dataset/devtest | wc -l

## Shared setup: language mapping, corpus reader, labels

In [ ]:
import io, re, json
import numpy as np, pandas as pd
from pathlib import Path

SEED = 42
RNG = np.random.default_rng(SEED)

LANG2FLORES = {
    "albanian":"als_Latn","asante":"twi_Latn","assamese":"asm_Beng","azeri":"azj_Latn",
    "bosnian-mihanovich":"bos_Latn","bosnian-rwwad":"bos_Latn","chinese":"zho_Hans",
    "croatian":"hrv_Latn","dutch":"nld_Latn","english":"eng_Latn","french":"fra_Latn",
    "german":"deu_Latn","gujarati":"guj_Gujr","hausa":"hau_Latn","indonesian":"ind_Latn",
    "japanese":"jpn_Jpan","kannada":"kan_Knda","khmer":"khm_Khmr","kurdish":"ckb_Arab",
    "kyrgyz":"kir_Cyrl","lingala":"lin_Latn","lithuanian":"lit_Latn","macedonian":"mkd_Cyrl",
    "malayalam":"mal_Mlym","moore":"mos_Latn","oromo":"gaz_Latn","pashto":"pbt_Arab",
    "persian":"pes_Arab","portuguese":"por_Latn","romanian":"ron_Latn","serbian":"srp_Cyrl",
    "spanish":"spa_Latn","tagalog":"tgl_Latn","tajik":"tgk_Cyrl","tamil":"tam_Taml",
    "turkish":"tur_Latn","urdu":"urd_Arab","uyghur":"uig_Arab","uzbek":"uzn_Latn",
    "vietnamese":"vie_Latn","yoruba":"yor_Latn","arabic_source":"arb_Arab",
}
MEDINAN = {2,3,4,5,8,9,13,22,24,33,47,48,49,55,57,58,59,60,61,62,63,64,65,66,76,98,99,110}

def lang_key(fname):
    if fname.startswith("bosnian_mihanovich"): return "bosnian-mihanovich"
    if fname.startswith("bosnian_rwwad"): return "bosnian-rwwad"
    return fname.split("_")[0]

def read_corpus_csv(path):
    txt = Path(path).read_text(encoding="utf-8-sig", errors="replace")
    m = re.search(r"^id,sura,aya", txt, flags=re.M)
    df = pd.read_csv(io.StringIO(txt[m.start():]))[["sura","aya","translation"]].dropna()
    df["sura"] = df.sura.astype(int); df["aya"] = df.aya.astype(int)
    return df

FILES = {lang_key(f.name): f for f in sorted(Path("corpus").glob("*.csv"))}
print(len(FILES), "corpus languages")

## A. FLORES-200 fertility replication

In [ ]:
import tiktoken
from transformers import AutoTokenizer

def get_tokenizers():
    toks = {}
    o200k = tiktoken.get_encoding("o200k_base"); cl100k = tiktoken.get_encoding("cl100k_base")
    toks["gpt4o_o200k"] = lambda s: len(o200k.encode(s, disallowed_special=()))
    toks["gpt4_cl100k"] = lambda s: len(cl100k.encode(s, disallowed_special=()))
    hf = {"mbert": ["bert-base-multilingual-cased"],
          "xlmr": ["xlm-roberta-base"],
          "bloom": ["bigscience/bloom-560m"],
          "nllb": ["facebook/nllb-200-distilled-600M"],
          "mt5": ["google/mt5-small"],                       # Aya-101 row == mT5 row
          "llama3": ["meta-llama/Meta-Llama-3-8B", "unsloth/llama-3-8b"],
          "gemma2": ["google/gemma-2-2b", "unsloth/gemma-2-2b"]}
    for key, names in hf.items():
        for n in names:
            try:
                t = AutoTokenizer.from_pretrained(n)
                toks[key] = (lambda tt: (lambda s: len(tt.encode(s, add_special_tokens=False))))(t)
                print("loaded", key, "<-", n); break
            except Exception as e:
                print("  skip", n, type(e).__name__)
    toks["byt5_bytes"] = lambda s: len(s.encode("utf-8"))
    return toks

TOKS = get_tokenizers()

In [ ]:
codes = sorted(set(LANG2FLORES.values()))
sents = {c: Path(f"flores200_dataset/devtest/{c}.devtest").read_text(encoding="utf-8").rstrip("\n").split("\n") for c in codes}
rows = []
for tk, fn in TOKS.items():
    for c in codes:
        rows.append({"tokenizer": tk, "flores_code": c,
                     "mean_tok_per_sent": float(np.mean([fn(s) for s in sents[c]]))})
    print("done", tk)
flores = pd.DataFrame(rows)
flores.to_csv("flores_fertility_by_language.csv", index=False)

In [ ]:
# Compare with the corpus audit (upload fertility_production_v2_all11.csv or mount Drive)
from scipy.stats import spearmanr
CORPUS_COL = {"gpt4o_o200k":"gpt4o_o200k_tok_per_verse","gpt4_cl100k":"gpt4_cl100k_tok_per_verse",
              "llama3":"llama3_tok_per_verse","gemma2":"gemma_tok_per_verse",
              "mbert":"mbert_tok_per_verse","xlmr":"xlmr_tok_per_verse",
              "bloom":"bloom_tok_per_verse","nllb":"nllb_tok_per_verse",
              "mt5":"mt5_tok_per_verse","byt5_bytes":"byt5_tok_per_verse"}
corpus = pd.read_csv("fertility_production_v2_all11.csv")
corpus = corpus[corpus.language.isin(LANG2FLORES)].copy()
corpus["flores_code"] = corpus.language.map(LANG2FLORES)

def ineq(v):
    v = np.asarray(v, float)
    return np.percentile(v,90)/np.percentile(v,10), v.max()/v.min()

summary = []
for tk, col in CORPUS_COL.items():
    if tk not in TOKS: continue
    cg = corpus.groupby("flores_code")[col].mean()
    fg = flores[flores.tokenizer==tk].set_index("flores_code")["mean_tok_per_sent"]
    common = sorted(set(cg.index) & set(fg.index))
    rho, p = spearmanr(cg[common], fg[common])
    (qp,qs),(fp,fs) = ineq(cg[common]), ineq(fg[common])
    summary.append({"tokenizer":tk,"corpus_p90p10":round(qp,2),"flores_p90p10":round(fp,2),
                    "corpus_maxmin":round(qs,2),"flores_maxmin":round(fs,2),
                    "spearman_lang":round(rho,3),"p":f"{p:.1e}"})
sm = pd.DataFrame(summary).sort_values("flores_p90p10")
print(sm.to_string(index=False))
rho_rank, p_rank = spearmanr(sm.corpus_p90p10, sm.flores_p90p10)
print(f"\nTokenizer inequality rank agreement: rho={rho_rank:.3f} p={p_rank:.2e}")
sm.to_csv("flores_vs_corpus_summary.csv", index=False)

## B. Downstream zero-shot transfer (Meccan vs Medinan)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

N_EVAL, N_TRAIN = 1000, 2000
en = read_corpus_csv(FILES["english"])
keys = list(en[["sura","aya"]].itertuples(index=False, name=None))
idx = RNG.permutation(len(keys))
eval_keys  = [keys[i] for i in sorted(idx[:N_EVAL])]
train_keys = [keys[i] for i in sorted(idx[N_EVAL:N_EVAL+N_TRAIN])]
lab = lambda ks: np.array([1 if s in MEDINAN else 0 for s,a in ks])
y_eval, y_train = lab(eval_keys), lab(train_keys)

def pick(df, ks):
    d = df.set_index(["sura","aya"])["translation"]
    return [str(d.get(k,"")) for k in ks]

ENCODERS = {"labse":"sentence-transformers/LaBSE",
            "mpnet":"sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
            "e5":"intfloat/multilingual-e5-base",
            "bgem3":"BAAI/bge-m3"}

all_rows = []
for tag, name in ENCODERS.items():
    model = SentenceTransformer(name)
    prefix = "query: " if "e5" in name else ""
    emb = lambda ts: model.encode([prefix+t for t in ts], batch_size=128,
                                  convert_to_numpy=True, normalize_embeddings=True)
    clf = LogisticRegression(max_iter=3000).fit(emb(pick(en, train_keys)), y_train)
    for lang, f in sorted(FILES.items()):
        pred = clf.predict(emb(pick(read_corpus_csv(f), eval_keys)))
        all_rows.append({"encoder":tag,"language":lang,
                         "accuracy":round(float(accuracy_score(y_eval,pred)),4),
                         "macro_f1":round(float(f1_score(y_eval,pred,average="macro")),4)})
        print(all_rows[-1], flush=True)
    del model
down = pd.DataFrame(all_rows)
down.to_csv("downstream_all_encoders.csv", index=False)

In [ ]:
# Which axis predicts the downstream transfer gap? (upload joint_language_table.csv)
joint = pd.read_csv("joint_language_table.csv")
for tag in down.encoder.unique():
    m = down[down.encoder==tag].merge(joint, on="language")
    for axis in ["retrieval","fertility","joshi"]:
        rho, p = spearmanr(m.macro_f1, m[axis])
        print(f"{tag:6s} macro-F1 vs {axis:9s}: rho={rho:+.3f}  p={p:.2e}")
    print()

In [ ]:
# Figure: transfer macro-F1 vs retrieval axis, colored by Joshi tier
import matplotlib.pyplot as plt
m = down[down.encoder=="e5"].merge(joint, on="language")
fig, ax = plt.subplots(figsize=(7,5))
sc = ax.scatter(m.retrieval, m.macro_f1, c=m.joshi, cmap="viridis", s=60)
for _, r in m.iterrows():
    if r.macro_f1 < 0.55 or r.retrieval < 0.35:
        ax.annotate(r.language, (r.retrieval, r.macro_f1), fontsize=8,
                    xytext=(4,4), textcoords="offset points")
ax.set_xlabel("Cross-lingual retrieval accuracy (mean top-1)")
ax.set_ylabel("Zero-shot Meccan/Medinan macro-F1")
plt.colorbar(sc, label="Joshi resource tier")
plt.tight_layout(); plt.savefig("downstream_vs_retrieval.png", dpi=200); plt.show()